### Get ddG values per split as with ML data

In [1]:
""""
Copyright (C) 2025 ETH Zurich, Stefan J.P. Chrin, Gregory Landrum, Sereina Riniker
"""

import pandas as pd
import json

# ligand/pair ids repeat across targets, and the two tables use different
# target vocabularies, so matching must be on (target, name).
TARGET_NAME_MAP = {"hunt": "bace_hunt", "p2": "bace_p2"}


# load FEP DDG data
fep_df = pd.read_csv('OPLS_FEP_DDG_FROM_GRAPH.csv')

# Load the JSON file containing which pairs are in which splits
with open('../../../2D/STRAT/B_DDG_FROM_DG/ddG_pred_&_error_all_splits.json', 'r') as f:
    splits = json.load(f)

# assume `splits` is your list of dicts
for split in splits:
    for target, pair_dict in split.items():
        for pair_name in sorted(pair_dict):            # sorted for reproducibility
            row = fep_df[(fep_df['target'] == TARGET_NAME_MAP.get(target, target))
                         & (fep_df['pair'] == pair_name)]
            # one row per shortest path; DDG is the path-average, identical within a group
            if len(row) < 1 or row['DDG'].nunique() != 1:
                raise ValueError(f"Bad FEP match for target '{target}' pair '{pair_name}': {len(row)} rows, {row['DDG'].nunique()} values")
            ddg_val = row['DDG'].iloc[0]
            pair_dict[pair_name]['prediction'] = ddg_val

# save updated splits
with open('OPLS_STRAT_ddG_FROM_FEP_GRAPH_pred&error.json', 'w') as f:
    json.dump(splits, f, indent=2)


### Sanity Check

In [2]:
import pandas as pd

# load data
df = pd.read_csv('../../../../0_DATA/HAHN/03b_all_ligands_all_ffs.csv')

# filter for ligands
filtered = df[df['ligand'].isin(['3a', '8a'])][['ligand', 'DG_OPLS3e']]

print(filtered)


    ligand  DG_OPLS3e
50      3a  -7.798851
577     3a -12.814687
592     8a  -8.395932
